# KQML / KIF Agent Dialogue — Python Implementation

**MSc Artificial Intelligence · Agent-Based Systems activity**

Alice (a procurement agent) queries Bob (a warehouse stock agent) about 50-inch
televisions: first the available stock, then the number of HDMI slots.

The dialogue is implemented as real message-passing between two Python agents:

- **KQML** provides the outer *communication* layer — performatives (`advertise`,
  `ask-one`, `tell`, `sorry`) expressing the sender's intent as speech acts.
- **KIF** provides the inner *knowledge* layer — first-order-logic content such as
  `(available-stock television-50inch ?quantity)`.
- A shared **ontology** (`warehouse-stock`) supplies the semantics both agents
  commit to before exchanging any messages.

## 1 · KQML message structure

Each message is a performative plus parameters (`:sender`, `:receiver`,
`:reply-with`, `:in-reply-to`, …) wrapping a KIF content expression.
`render()` prints the classic s-expression syntax.

In [1]:
from dataclasses import dataclass, field
from typing import Optional
import re

@dataclass
class KQMLMessage:
    """A KQML message: performative + parameters + KIF content."""
    performative: str                 # e.g. ask-one, tell, sorry, advertise
    sender: str
    receiver: str
    content: str                      # KIF expression as a string
    language: str = "KIF"
    ontology: str = "warehouse-stock"
    reply_with: Optional[str] = None
    in_reply_to: Optional[str] = None

    def render(self) -> str:
        """Render the message in classic KQML s-expression syntax."""
        lines = [f"({self.performative}",
                 f"  :sender       {self.sender}",
                 f"  :receiver     {self.receiver}",
                 f"  :language     {self.language}",
                 f"  :ontology     {self.ontology}"]
        if self.reply_with:
            lines.append(f"  :reply-with   {self.reply_with}")
        if self.in_reply_to:
            lines.append(f"  :in-reply-to  {self.in_reply_to}")
        lines.append(f"  :content      {self.content})")
        return "\n".join(lines)

## 2 · Shared ontology: `warehouse-stock`

Without prior semantic agreement, a syntactically valid message would still be
meaningless — the ontology is what makes `hdmi-slots` mean the same thing to
both agents. Bob validates every incoming query against it.

In [2]:
ONTOLOGY = {
    "name": "warehouse-stock",
    "classes": ["Product", "Television"],
    "functions": ["available-stock", "hdmi-slots", "screen-size"],
    "instances": ["television-50inch", "television-65inch", "soundbar-basic"],
}

def parse_kif_query(content: str):
    """Minimal KIF query parser:  (function instance ?variable)"""
    m = re.match(r"\((\S+)\s+(\S+)\s+\?\S+\)", content.strip())
    return (m.group(1), m.group(2)) if m else None

## 3 · Bob — the warehouse agent

Bob owns a private knowledge base. He handles `ask-one` by parsing the KIF
content, checking the function exists in the ontology, and replying with
`tell` (a fact) or `sorry` (graceful failure). Alice never sees how his data
is stored — that encapsulation is the point of knowledge-level communication.

In [3]:
@dataclass
class WarehouseAgent:
    """Bob — controls stock levels; answers KIF queries against his KB."""
    name: str = "Bob"
    knowledge_base: dict = field(default_factory=lambda: {
        ("available-stock", "television-50inch"): 137,
        ("hdmi-slots",      "television-50inch"): 3,
        ("screen-size",     "television-50inch"): 50,
        ("available-stock", "television-65inch"): 42,
        ("hdmi-slots",      "television-65inch"): 4,
        ("available-stock", "soundbar-basic"):    260,
    })

    def advertise(self, to: str) -> KQMLMessage:
        return KQMLMessage(
            performative="advertise", sender=self.name, receiver=to,
            language="KQML",
            content="(ask-one :content (?function ?product ?value))")

    def handle(self, msg: KQMLMessage) -> KQMLMessage:
        """Process an incoming ask-one; reply with tell or sorry."""
        if msg.performative != "ask-one":
            return self._sorry(msg, "unsupported performative")
        parsed = parse_kif_query(msg.content)
        if parsed is None:
            return self._sorry(msg, "cannot parse KIF content")
        function, instance = parsed
        if function not in ONTOLOGY["functions"]:
            return self._sorry(msg, f"unknown function {function} in ontology")
        value = self.knowledge_base.get((function, instance))
        if value is None:
            return self._sorry(msg, f"no fact for ({function} {instance})")
        return KQMLMessage(
            performative="tell", sender=self.name, receiver=msg.sender,
            in_reply_to=msg.reply_with,
            content=f"(= ({function} {instance}) {value})")

    def _sorry(self, msg: KQMLMessage, reason: str) -> KQMLMessage:
        return KQMLMessage(
            performative="sorry", sender=self.name, receiver=msg.sender,
            in_reply_to=msg.reply_with,
            content=f'(reason "{reason}")')

## 4 · Alice — the procurement agent

Alice builds `ask-one` queries (she wants a single binding, not all matching
facts, which would be `ask-all`) and maintains a **belief base** that she
updates whenever a `tell` arrives — a minimal version of the belief component
in BDI-style architectures.

In [4]:
@dataclass
class ProcurementAgent:
    """Alice — needs product facts before purchasing; builds ask-one queries."""
    name: str = "Alice"
    _counter: int = 0
    beliefs: dict = field(default_factory=dict)

    def ask_one(self, to: str, function: str, instance: str,
                variable: str) -> KQMLMessage:
        self._counter += 1
        return KQMLMessage(
            performative="ask-one", sender=self.name, receiver=to,
            reply_with=f"query-{instance}-{self._counter:02d}",
            content=f"({function} {instance} ?{variable})")

    def receive(self, msg: KQMLMessage) -> str:
        """Update beliefs from a tell; report what was learned."""
        if msg.performative == "tell":
            m = re.match(r"\(=\s*\((\S+)\s+(\S+)\)\s+(\S+)\)", msg.content)
            if m:
                function, instance, value = m.groups()
                self.beliefs[(function, instance)] = value
                return f"{self.name} now believes ({function} {instance}) = {value}"
        if msg.performative == "sorry":
            return f"{self.name} notes the query failed: {msg.content}"
        return f"{self.name} ignored a {msg.performative} message"

## 5 · Running the dialogue from the activity

The five-message exchange: Bob advertises, Alice asks about available stock of
50-inch televisions, Bob tells her (137 units), Alice asks about HDMI slots,
Bob tells her (3 slots).

In [5]:
def run_dialogue(product: str = "television-50inch"):
    """Yields (step_label, KQMLMessage, note) tuples for the full exchange."""
    alice, bob = ProcurementAgent(), WarehouseAgent()

    adv = bob.advertise(alice.name)
    yield ("0 · advertise", adv, "Bob announces his query-answering capability")

    q1 = alice.ask_one(bob.name, "available-stock", product, "quantity")
    yield ("1 · ask-one", q1, "Alice queries the available stock")
    r1 = bob.handle(q1)
    yield ("2 · " + r1.performative, r1, alice.receive(r1))

    q2 = alice.ask_one(bob.name, "hdmi-slots", product, "number")
    yield ("3 · ask-one", q2, "Alice queries the number of HDMI slots")
    r2 = bob.handle(q2)
    yield ("4 · " + r2.performative, r2, alice.receive(r2))

for label, message, note in run_dialogue():
    print(f"--- {label} " + "-" * (60 - len(label)))
    print(message.render())
    print(f">>> {note}\n")

--- 0 · advertise -----------------------------------------------
(advertise
  :sender       Bob
  :receiver     Alice
  :language     KQML
  :ontology     warehouse-stock
  :content      (ask-one :content (?function ?product ?value)))
>>> Bob announces his query-answering capability

--- 1 · ask-one -------------------------------------------------
(ask-one
  :sender       Alice
  :receiver     Bob
  :language     KIF
  :ontology     warehouse-stock
  :reply-with   query-television-50inch-01
  :content      (available-stock television-50inch ?quantity))
>>> Alice queries the available stock

--- 2 · tell ----------------------------------------------------
(tell
  :sender       Bob
  :receiver     Alice
  :language     KIF
  :ontology     warehouse-stock
  :in-reply-to  query-television-50inch-01
  :content      (= (available-stock television-50inch) 137))
>>> Alice now believes (available-stock television-50inch) = 137

--- 3 · ask-one ------------------------------------------------